<a href="https://colab.research.google.com/github/yosie111/Shulchan_Aruch_RAG_1/blob/main/SA_RAG_Stage3_Reranker.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Shulchan Aruch RAG — Stage 3: Reranker Benchmark (v2)

## ממצאי Stage 2 (K=40):
```
                K=5    K=20   K=25   K=40    ← ceiling
bge-m3_vector   79.4%  90.2%  90.2%  91.2%
bge-m3_hybrid   75.5%  90.2%  94.1%  94.1%
e5-large_hybrid 78.4%  89.2%  95.1%  95.1%   ← BEST ceiling
```

## מה נבדק כאן:
```
1. bge-m3 vector top-20         (baseline ישן)
2. bge-m3 vector top-20 + RR    (Stage 3 ישן — הגיע ל-87.3%)
3. bge-m3 hybrid top-25 + RR    (ceiling 94.1%)
4. e5-large hybrid top-25 + RR  (ceiling 95.1% — צפי הכי גבוה)
5. e5-large vector top-25 + RR  (בקרה)
```

## הוראות:
- Runtime → GPU
- העלה JSONL + Excel
- הרץ תאים 1-7
- **לא צריך API key**


In [1]:
# ============================================================
# תא 1 — התקנה + ייבוא
# ============================================================
!pip install -q sentence-transformers chromadb openpyxl rank-bm25

import json
import re
import os
import time
import numpy as np
from pathlib import Path
from collections import defaultdict

import openpyxl
import chromadb
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi

try:
    from google.colab import files as colab_files
    IN_COLAB = True
except Exception:
    IN_COLAB = False

import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
print('✅ תא 1')


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 108.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 102.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 101.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 34.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 9.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not cu

In [2]:
# ============================================================
# תא 2 — טעינת נתונים + שאלות evaluation
# ============================================================

# --- A. JSONL ---
JSONL_FILE = None
for p in sorted(Path('.').glob('*.jsonl')):
    JSONL_FILE = str(p); break
if JSONL_FILE is None and IN_COLAB:
    print('העלה JSONL:'); uploaded = colab_files.upload()
    for n in uploaded:
        if n.endswith('.jsonl'): JSONL_FILE = n; break
if not JSONL_FILE: raise FileNotFoundError('JSONL not found')

chunks = []
with open(JSONL_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip(): chunks.append(json.loads(line))
content_chunks = [c for c in chunks if c.get('metadata',{}).get('doc_type') != 'hilchot_index']
print(f'📄 {len(content_chunks)} chunks')

# --- B. Excel ---
EVAL_FILE = None
for p in sorted(Path('.').glob('*.xlsx')):
    EVAL_FILE = str(p); break
if EVAL_FILE is None and IN_COLAB:
    print('העלה Excel:'); uploaded = colab_files.upload()
    for n in uploaded:
        if n.endswith('.xlsx'): EVAL_FILE = n; break
if not EVAL_FILE: raise FileNotFoundError('Excel not found')

GEMATRIA = {'א':1,'ב':2,'ג':3,'ד':4,'ה':5,'ו':6,'ז':7,'ח':8,'ט':9,
    'י':10,'כ':20,'ל':30,'מ':40,'נ':50,'ס':60,'ע':70,'פ':80,'צ':90,
    'ק':100,'ר':200,'ש':300,'ת':400,'ך':20,'ם':40,'ן':50,'ף':80,'ץ':90}
def heb2num(s):
    return sum(GEMATRIA.get(c,0) for c in re.sub(r'["\'\'  ׳״]','',s or ''))

eval_questions = []
wb = openpyxl.load_workbook(EVAL_FILE); ws = wb.active
for row in range(2, ws.max_row + 1):
    q = ws.cell(row,1).value
    if not q: continue
    src = ws.cell(row,3).value or ''
    sm = re.search(r'סימן\s+([א-ת]+)', src)
    sem = re.search(r'סעיף\s+([א-ת]+)', src)
    eval_questions.append({
        'question': q, 'answer': ws.cell(row,2).value or '',
        'source': src,
        'gold_siman': heb2num(sm.group(1)) if sm else 0,
        'gold_seif': heb2num(sem.group(1)) if sem else 0,
    })
print(f'📝 {len(eval_questions)} שאלות')

# --- C. Gold mapping ---
def find_gold_chunk(q, chs):
    gs, gse = q['gold_siman'], q['gold_seif']
    for c in chs:
        h = c.get('metadata',{}).get('hierarchy',{})
        if h.get('level_1_num') == gs:
            if gse in h.get('level_2_nums',[]) or gse == 0:
                return c['doc_id']
    for c in chs:
        if c.get('metadata',{}).get('hierarchy',{}).get('level_1_num') == gs:
            return c['doc_id']
    return None

for q in eval_questions:
    q['gold_chunk_id'] = find_gold_chunk(q, content_chunks)
valid_questions = [q for q in eval_questions if q['gold_chunk_id']]
print(f'📌 {len(valid_questions)} שאלות עם gold mapping')

# --- D. Shared data ---
chunk_ids = [c['doc_id'] for c in content_chunks]
chunk_texts_emb = [c['content']['text_for_embedding'] for c in content_chunks]
chunk_texts_bm25 = [c['content']['text_for_bm25'] for c in content_chunks]
chunk_texts_llm = [c['content']['text_for_llm'] for c in content_chunks]
id_to_llm = {c['doc_id']: c['content']['text_for_llm'] for c in content_chunks}

print(f'✅ תא 2')


העלה JSONL:


Saving shulchan_aruch_v2 (11).jsonl to shulchan_aruch_v2 (11).jsonl
📄 1355 chunks
העלה Excel:


Saving על שוע    חלק א.xlsx to על שוע    חלק א.xlsx
📝 102 שאלות
📌 102 שאלות עם gold mapping
✅ תא 2


In [3]:
# ============================================================
# תא 3 — טעינת 2 מודלי embedding + BM25 + ChromaDB
# ============================================================

EMBEDDING_MODELS = {
    'bge-m3': 'BAAI/bge-m3',
    'e5-large': 'intfloat/multilingual-e5-large',
}

# e5 דורש prefix
E5_DOC_PREFIX = 'passage: '
E5_QUERY_PREFIX = 'query: '

chroma = chromadb.Client()
embed_models = {}  # name → {'model': ..., 'collection': ...}

for name, path in EMBEDDING_MODELS.items():
    print(f'\nLoading {name}...')
    t0 = time.time()
    model = SentenceTransformer(path, device=DEVICE)
    print(f'  Model loaded in {time.time()-t0:.1f}s')

    # Encode
    t1 = time.time()
    if name == 'e5-large':
        texts = [E5_DOC_PREFIX + t for t in chunk_texts_emb]
    else:
        texts = chunk_texts_emb
    vectors = model.encode(texts, show_progress_bar=True, normalize_embeddings=True)
    print(f'  Encoded {vectors.shape} in {time.time()-t1:.1f}s')

    # ChromaDB
    col_name = f'sa_{name}'.replace('-', '_')
    try: chroma.delete_collection(col_name)
    except: pass
    col = chroma.create_collection(col_name, metadata={'hnsw:space': 'cosine'})
    col.add(
        ids=chunk_ids,
        embeddings=[v.tolist() for v in vectors],
        documents=chunk_texts_llm,
    )
    embed_models[name] = {'model': model, 'collection': col}
    print(f'  ChromaDB: {col.count()} vectors')

# BM25
print('\nBuilding BM25...')
tokenized_corpus = [t.split() for t in chunk_texts_bm25]
bm25_index = BM25Okapi(tokenized_corpus)
print(f'  {len(tokenized_corpus)} documents indexed')

print(f'\n✅ תא 3 — 2 embedding models + BM25')



Loading bge-m3...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/201 [00:00<?, ?B/s]

  Model loaded in 11.9s


Batches:   0%|          | 0/43 [00:00<?, ?it/s]

  Encoded (1355, 1024) in 6.7s
  ChromaDB: 1355 vectors

Building BM25...
  1355 documents indexed

✅ תא 3 — 2 embedding models + BM25


In [4]:
# ============================================================
# תא 4 — Reranker + פונקציות retrieval
# ============================================================

# Reranker (רק bge — mxbai נפסל ב-Stage 3 v1)
print('Loading bge-reranker-v2-m3...')
t0 = time.time()
reranker = CrossEncoder('BAAI/bge-reranker-v2-m3', device=DEVICE)
print(f'  Loaded in {time.time()-t0:.1f}s')


RRF_K = 60

def retrieve_vector(question, embed_name, n=25):
    """bi-encoder בלבד"""
    model = embed_models[embed_name]['model']
    col = embed_models[embed_name]['collection']
    q_text = (E5_QUERY_PREFIX + question) if embed_name == 'e5-large' else question
    qvec = model.encode([q_text], normalize_embeddings=True)
    res = col.query(query_embeddings=[qvec[0].tolist()], n_results=n)
    return [(cid, id_to_llm.get(cid, '')) for cid in res['ids'][0]]


def retrieve_hybrid(question, embed_name, n=25, vector_weight=0.7):
    """vector + BM25 RRF"""
    # Vector
    model = embed_models[embed_name]['model']
    col = embed_models[embed_name]['collection']
    q_text = (E5_QUERY_PREFIX + question) if embed_name == 'e5-large' else question
    qvec = model.encode([q_text], normalize_embeddings=True)
    vec_res = col.query(query_embeddings=[qvec[0].tolist()], n_results=n)
    vec_ids = vec_res['ids'][0]

    # BM25
    bm25_scores = bm25_index.get_scores(question.split())
    bm25_top = np.argsort(bm25_scores)[::-1][:n]
    bm25_ids = [chunk_ids[i] for i in bm25_top]

    # RRF
    rrf = defaultdict(float)
    for rank, cid in enumerate(vec_ids):
        rrf[cid] += vector_weight / (RRF_K + rank + 1)
    for rank, cid in enumerate(bm25_ids):
        rrf[cid] += (1 - vector_weight) / (RRF_K + rank + 1)

    sorted_ids = sorted(rrf, key=rrf.get, reverse=True)[:n]
    return [(cid, id_to_llm.get(cid, '')) for cid in sorted_ids]


def do_rerank(question, candidates, top_k=10):
    """cross-encoder reranking"""
    if not candidates: return []
    pairs = [(question, text) for _, text in candidates]
    scores = reranker.predict(pairs)
    scored = sorted(zip(candidates, scores), key=lambda x: x[1], reverse=True)
    return [(cid, text, float(sc)) for (cid, text), sc in scored[:top_k]]


# Metrics
K_VALUES = [1, 3, 5, 10]

def recall_at_k(ids, gold, k):
    return 1.0 if gold in ids[:k] else 0.0

def mrr(ids, gold):
    for i, r in enumerate(ids):
        if r == gold: return 1.0 / (i + 1)
    return 0.0

def compute_metrics(all_ids, all_gold):
    m = {}
    for k in K_VALUES:
        m[f'Recall@{k}'] = np.mean([recall_at_k(r, g, k) for r, g in zip(all_ids, all_gold)])
    m['MRR'] = np.mean([mrr(r, g) for r, g in zip(all_ids, all_gold)])
    return m

print(f'\n✅ תא 4')


Loading bge-reranker-v2-m3...


config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

  Loaded in 12.9s

✅ תא 4


In [5]:
# ============================================================
# תא 5 — Benchmark: 6 קונפיגורציות
# ============================================================

TOP_N = 25  # retrieval ראשוני (25 כי e5-hybrid peaks at K=25)

gold_ids = [q['gold_chunk_id'] for q in valid_questions]
all_results = {}  # method → list of [chunk_ids]
all_candidates = {}  # method → list of [(cid, text)] for reranking

CONFIGS = [
    # (name, embed_model, retrieval_type, use_reranker)
    ('bge-m3 vector',              'bge-m3',   'vector', False),
    ('bge-m3 vector + RR',         'bge-m3',   'vector', True),
    ('bge-m3 hybrid + RR',         'bge-m3',   'hybrid', True),
    ('e5-large vector + RR',       'e5-large', 'vector', True),
    ('e5-large hybrid',            'e5-large', 'hybrid', False),
    ('e5-large hybrid + RR',       'e5-large', 'hybrid', True),
]

for cfg_idx, (name, embed, ret_type, use_rr) in enumerate(CONFIGS, 1):
    print(f'[{cfg_idx}/{len(CONFIGS)}] {name}...')
    t0 = time.time()

    ids_per_q = []
    cands_per_q = []

    for i, q in enumerate(valid_questions):
        # Retrieve
        if ret_type == 'vector':
            cands = retrieve_vector(q['question'], embed, n=TOP_N)
        else:
            cands = retrieve_hybrid(q['question'], embed, n=TOP_N)

        # Rerank?
        if use_rr:
            reranked = do_rerank(q['question'], cands, top_k=TOP_N)
            ids_per_q.append([c[0] for c in reranked])
        else:
            ids_per_q.append([c[0] for c in cands])

        cands_per_q.append(cands)

        if (i+1) % 25 == 0:
            print(f'    [{i+1}/{len(valid_questions)}]')

    all_results[name] = ids_per_q
    all_candidates[name] = cands_per_q
    print(f'    Done in {time.time()-t0:.1f}s')

# === Compute + Print ===
all_metrics = {}
for method, retrieved in all_results.items():
    all_metrics[method] = compute_metrics(retrieved, gold_ids)

print(f'\n{"="*85}')
print(f'{"RERANKER BENCHMARK v2 — 1355 chunks":^85}')
print(f'{"="*85}')

header = f'{"Method":<30}'
for k in K_VALUES: header += f'{"R@"+str(k):>8}'
header += f'{"MRR":>8}  {"Ceiling":>8}'
print(header)
print('-' * len(header))

# Ceiling = recall in the candidate pool before reranking
for method in [c[0] for c in CONFIGS]:
    m = all_metrics[method]
    # Ceiling: how many gold in top-N candidates?
    cands = all_candidates.get(method, all_results[method])
    candidate_ids = all_results[method]  # after rerank these are reranked
    # For ceiling, use the pre-rerank candidates
    pool_ids = [[c[0] for c in clist] for clist in all_candidates[method]]
    ceiling = np.mean([1.0 if g in p else 0.0 for g, p in zip(gold_ids, pool_ids)])

    row = f'{method:<30}'
    for k in K_VALUES: row += f'{m[f"Recall@{k}"]:>8.3f}'
    row += f'{m["MRR"]:>8.3f}  {ceiling:>7.1%}'
    print(row)

best_method = max(all_metrics, key=lambda x: all_metrics[x]['Recall@5'])
best_r5 = all_metrics[best_method]['Recall@5']
baseline_r5 = all_metrics['bge-m3 vector']['Recall@5']
print(f'\n🏆 Best: {best_method} (Recall@5={best_r5:.3f}, Δ from baseline={best_r5-baseline_r5:+.3f})')

print(f'\n✅ תא 5')


[1/6] bge-m3 vector...
    [25/102]
    [50/102]
    [75/102]
    [100/102]
    Done in 1.2s
[2/6] bge-m3 vector + RR...
    [25/102]
    [50/102]
    [75/102]
    [100/102]
    Done in 22.2s
[3/6] bge-m3 hybrid + RR...
    [25/102]
    [50/102]
    [75/102]
    [100/102]
    Done in 22.2s
[4/6] e5-large vector + RR...
    [25/102]
    [50/102]
    [75/102]
    [100/102]
    Done in 22.7s
[5/6] e5-large hybrid...
    [25/102]
    [50/102]
    [75/102]
    [100/102]
    Done in 1.3s
[6/6] e5-large hybrid + RR...
    [25/102]
    [50/102]
    [75/102]
    [100/102]
    Done in 23.3s

                         RERANKER BENCHMARK v2 — 1355 chunks                         
Method                             R@1     R@3     R@5    R@10     MRR   Ceiling
--------------------------------------------------------------------------------
bge-m3 vector                    0.608   0.745   0.794   0.853   0.689    90.2%
bge-m3 vector + RR               0.716   0.873   0.873   0.892   0.796    90.2%
bge

In [6]:
# ============================================================
# תא 6 — ניתוח per-שאלה: best vs baseline
# ============================================================

best_ids = all_results[best_method]
base_ids = all_results['bge-m3 vector']

fixed, broke, both_hit, both_miss = [], [], 0, 0

for i, q in enumerate(valid_questions):
    gold = q['gold_chunk_id']
    b_hit = gold in base_ids[i][:5]
    r_hit = gold in best_ids[i][:5]
    if b_hit and r_hit: both_hit += 1
    elif not b_hit and not r_hit: both_miss += 1
    elif not b_hit and r_hit:
        fixed.append({'q': q['question'], 'src': q['source'], 'gold': gold,
                       'base': base_ids[i][:3], 'best': best_ids[i][:3]})
    else:
        broke.append({'q': q['question'], 'src': q['source'], 'gold': gold,
                       'base': base_ids[i][:3], 'best': best_ids[i][:3]})

n = len(valid_questions)
print(f'{"="*70}')
print(f'{"PER-QUESTION ANALYSIS":^70}')
print(f'{"="*70}')
print(f'  baseline: bge-m3 vector | best: {best_method}')
print(f'  Both hit:  {both_hit:>3} ({100*both_hit//n}%)')
print(f'  Fixed:     {len(fixed):>3} ({100*len(fixed)//n}%) ← שיפור')
print(f'  Broke:     {len(broke):>3} ({100*len(broke)//n}%) ← נזק')
print(f'  Both miss: {both_miss:>3} ({100*both_miss//n}%)')
print(f'  Net gain:  {len(fixed)-len(broke):+d}')

if fixed:
    print(f'\n--- תוקנו ({len(fixed)}) ---')
    for f in fixed[:8]:
        print(f'  Q: {f["q"][:55]}')
        print(f'  Gold: {f["gold"]}')
        print(f'  Base: {f["base"]} ❌ → Best: {f["best"]} ✅')
        print()

if broke:
    print(f'--- נשברו ({len(broke)}) ---')
    for b in broke[:5]:
        print(f'  Q: {b["q"][:55]}')
        print(f'  Gold: {b["gold"]}')
        print(f'  Base: {b["base"]} ✅ → Best: {b["best"]} ❌')
        print()

# Error categorization
if both_miss:
    print(f'--- נכשלו בשניהם ({both_miss}) ---')
    for i, q in enumerate(valid_questions):
        gold = q['gold_chunk_id']
        if gold in base_ids[i][:5] or gold in best_ids[i][:5]: continue
        # In pool?
        pool = [c[0] for c in all_candidates[best_method][i]]
        in_pool = gold in pool
        tag = 'RANKING' if in_pool else 'RETRIEVAL'
        print(f'  [{tag}] {q["question"][:50]} | Gold: {gold}')

print(f'\n✅ תא 6')


                        PER-QUESTION ANALYSIS                         
  baseline: bge-m3 vector | best: e5-large vector + RR
  Both hit:   79 (77%)
  Fixed:      12 (11%) ← שיפור
  Broke:       2 (1%) ← נזק
  Both miss:   9 (8%)
  Net gain:  +10

--- תוקנו (12) ---
  Q: מה הדין במנעלים ללא קשירה ?
  Gold: sa_oc_002_1-6
  Base: ['sa_oc_317_2-6__b', 'sa_oc_614_1-4', 'sa_oc_317_1__a'] ❌ → Best: ['sa_oc_002_1-6', 'sa_oc_317_2-6__b', 'sa_oc_317_1__a'] ✅

  Q: באיזו יד נוטל את כלי המים ?
  Gold: sa_oc_004_1-11__a
  Base: ['sa_oc_159_6__c', 'sa_oc_159_14-20__f', 'sa_oc_162_8-10__e'] ❌ → Best: ['sa_oc_004_1-11__a', 'sa_oc_159_8-13__e', 'sa_oc_159_14-20__f'] ✅

  Q: האם מותר ליטול ממי שלא נטל ידיו ?
  Gold: sa_oc_004_1-11__a
  Base: ['sa_oc_159_6__c', 'sa_oc_004_19-23__c', 'sa_oc_164_1-2'] ❌ → Best: ['sa_oc_004_1-11__a', 'sa_oc_163_1-2', 'sa_oc_004_19-23__c'] ✅

  Q: מה הרפואה לפנים מתבקעות ?
  Gold: sa_oc_004_19-23__c
  Base: ['sa_oc_230_4-5__b', 'sa_oc_575_1-3__a', 'sa_oc_053_6-10__b'] ❌ → B

In [7]:
# ============================================================
# תא 7 — שמירה + המלצה + הורדה
# ============================================================

OUTPUT_DIR = '/content/reranker_output' if IN_COLAB else 'reranker_output'
os.makedirs(OUTPUT_DIR, exist_ok=True)

delta = best_r5 - baseline_r5
if delta > 0.02:
    rec = 'USE'
    rec_text = f'USE {best_method} — Recall@5 {baseline_r5:.3f} → {best_r5:.3f} (Δ={delta:+.3f})'
elif delta >= 0:
    rec = 'OPTIONAL'
    rec_text = f'OPTIONAL — marginal improvement (Δ={delta:+.3f})'
else:
    rec = 'SKIP'
    rec_text = f'SKIP — no improvement'

print(f'{"="*70}')
print(f'{"RECOMMENDATION FOR STAGE 4":^70}')
print(f'{"="*70}')
print(f'  Baseline:  bge-m3 vector     Recall@5 = {baseline_r5:.3f}')
print(f'  Best:      {best_method:<22} Recall@5 = {best_r5:.3f}')
print(f'  Delta:     {delta:+.3f}')
print(f'  Fixed: {len(fixed)} | Broke: {len(broke)} | Net: {len(fixed)-len(broke):+d}')
print(f'')
print(f'  ➡️  {rec_text}')
print(f'')

# Architecture recommendation
if 'hybrid' in best_method and 'RR' in best_method:
    embed_name = 'e5-large' if 'e5' in best_method else 'bge-m3'
    print(f'  Architecture for Stage 4:')
    print(f'    שאלה → {embed_name} hybrid (top-{TOP_N}) → bge-reranker (top-5) → LLM')
elif 'vector' in best_method and 'RR' in best_method:
    embed_name = 'e5-large' if 'e5' in best_method else 'bge-m3'
    print(f'  Architecture for Stage 4:')
    print(f'    שאלה → {embed_name} vector (top-{TOP_N}) → bge-reranker (top-5) → LLM')
else:
    print(f'  Architecture for Stage 4:')
    print(f'    שאלה → {best_method} (top-5) → LLM')
print(f'{"="*70}')

# --- Save JSON ---
output = {
    'config': {
        'embedding_models': list(EMBEDDING_MODELS.keys()),
        'reranker': 'BAAI/bge-reranker-v2-m3',
        'top_n': TOP_N,
        'n_questions': len(valid_questions),
        'n_chunks': len(content_chunks),
    },
    'metrics': {m: {k: round(v,4) if isinstance(v,float) else v
                    for k,v in met.items()}
               for m, met in all_metrics.items()},
    'recommendation': rec,
    'best_method': best_method,
    'delta': round(delta, 4),
    'fixed': len(fixed), 'broke': len(broke),
    'fixed_questions': fixed[:20],
    'broke_questions': broke[:20],
}
json_path = os.path.join(OUTPUT_DIR, 'reranker_results.json')
with open(json_path, 'w', encoding='utf-8') as f:
    json.dump(output, f, ensure_ascii=False, indent=2)

# --- Save CSV ---
csv_path = os.path.join(OUTPUT_DIR, 'reranker_comparison.csv')
with open(csv_path, 'w', encoding='utf-8-sig') as f:
    h = 'Method'
    for k in K_VALUES: h += f',Recall@{k}'
    h += ',MRR'
    f.write(h + '\n')
    for method in sorted(all_metrics, key=lambda x: -all_metrics[x]['Recall@5']):
        m = all_metrics[method]
        row = method
        for k in K_VALUES: row += f',{m[f"Recall@{k}"]:.4f}'
        row += f',{m["MRR"]:.4f}'
        f.write(row + '\n')

# --- Save per-question ---
detail = []
for i, q in enumerate(valid_questions):
    gold = q['gold_chunk_id']
    detail.append({
        'question': q['question'], 'source': q['source'], 'gold': gold,
        'base_hit5': gold in base_ids[i][:5],
        'best_hit5': gold in best_ids[i][:5],
        'base_top5': base_ids[i][:5],
        'best_top5': best_ids[i][:5],
    })
detail_path = os.path.join(OUTPUT_DIR, 'per_question_detail.json')
with open(detail_path, 'w', encoding='utf-8') as f:
    json.dump(detail, f, ensure_ascii=False, indent=2)

# --- Save summary ---
summary_path = os.path.join(OUTPUT_DIR, 'reranker_summary.txt')
with open(summary_path, 'w', encoding='utf-8') as f:
    f.write(f'Reranker Benchmark v2 Summary\n{"="*50}\n')
    f.write(f'Chunks: {len(content_chunks)}\nQuestions: {len(valid_questions)}\n')
    f.write(f'Recommendation: {rec}\nBest: {best_method}\n')
    f.write(f'Recall@5: {baseline_r5:.3f} → {best_r5:.3f} (Δ={delta:+.3f})\n')
    f.write(f'Fixed: {len(fixed)} | Broke: {len(broke)} | Net: {len(fixed)-len(broke):+d}\n')

# --- List + Download ---
print(f'\n📂 {os.path.abspath(OUTPUT_DIR)}/')
for fname in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, fname)
    print(f'  📄 {fname} ({os.path.getsize(fpath):,} bytes)')

if IN_COLAB:
    print('\n⬇️  מוריד...')
    from google.colab import files as colab_files
    for fname in sorted(os.listdir(OUTPUT_DIR)):
        colab_files.download(os.path.join(OUTPUT_DIR, fname))
    print('✅ הורדו!')
else:
    print('\n✅ סיום')


                      RECOMMENDATION FOR STAGE 4                      
  Baseline:  bge-m3 vector     Recall@5 = 0.794
  Best:      e5-large vector + RR   Recall@5 = 0.892
  Delta:     +0.098
  Fixed: 12 | Broke: 2 | Net: +10

  ➡️  USE e5-large vector + RR — Recall@5 0.794 → 0.892 (Δ=+0.098)

  Architecture for Stage 4:
    שאלה → e5-large vector (top-25) → bge-reranker (top-5) → LLM

📂 /content/reranker_output/
  📄 per_question_detail.json (51,468 bytes)
  📄 reranker_comparison.csv (371 bytes)
  📄 reranker_results.json (6,397 bytes)
  📄 reranker_summary.txt (226 bytes)

⬇️  מוריד...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ הורדו!
